In [2]:
import numpy as np
from scipy import interpolate
import plotly.graph_objects as go

# Making some fake data to represent a flown profile

In [3]:
def generate_fake_profile():
    vertices = np.array([[30, 10], [15, 5], [90, -25], [180, 5], [140, 15]])
    x = vertices[:, 0]
    y = vertices[:, 1]
    t, u = interpolate.splprep([x, y], s=0)
    xnew, ynew = interpolate.splev(np.linspace(0, 1, 100), t, der=0)
    return xnew, ynew

In [4]:
xnew, ynew = generate_fake_profile()

In [5]:
def tangent_line(pt1: tuple, pt2: tuple, x_range: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    """ returns xs, ys of tangent line endpoints """
    # Calculate slope and intercept
    x1, y1 = pt1
    x2, y2 = pt2
    m = (y2 - y1) / (x2 - x1)
    b = y1 - m * x1

    # Calculate tangent line endpoints
    x_ave = (x2 + x1) / 2
    domain_size = x_range[1] - x_range[0]
    x1_end = x_ave - 0.20 * domain_size
    x2_end = x_ave + 0.20 * domain_size
    y1_end = m*x1_end + b
    y2_end = m*x2_end + b

    xs = np.array([x1_end, x2_end])
    ys = np.array([y1_end, y2_end])
    return xs, ys

# Visualizing the profile and its first derivative

In [6]:
my_frames = []
for idx in range(len(xnew)-1):
    pt1 = xnew[idx], ynew[idx]
    pt2 = xnew[idx+1], ynew[idx+1]
    xs, ys = tangent_line(pt1, pt2, x_range=(0, 180))

    frame = go.Frame(data=[go.Scatter(x=xs, y=ys, mode='lines', line=dict(color='red'))])
    my_frames.append(frame)

In [83]:
fig = go.Figure(
    data=[
        go.Scatter(x=xnew, y=ynew, mode='lines', line=dict(width=2, color='blue')),
        go.Scatter(x=xnew, y=ynew, mode='lines', line=dict(width=2, color='blue'), showlegend=False)
    ],
    layout=go.Layout(
        xaxis=dict(range=[0, 200], autorange=False),
        yaxis=dict(range=[-45, 30], autorange=False),
        title="Fake Profile",
        updatemenus=[dict(
            type="buttons",
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[
                         None,
                         {
                             "frame": {"duration": 10, "redraw": False},
                             "mode": "immediate",
                             "transition": {"duration": 10}
                         }
                     ])
            ]
        )]
    ),
    frames=my_frames
)
fig.show()

In [8]:
import plotly.offline as po
# po.plot(fig)

# Figuring out the transition points analytically

Make a pandas.DataFrame

In [29]:
import pandas as pd

In [56]:
data = pd.DataFrame(data=dict(Look=xnew, Depression=ynew))

In [57]:
data.head()

,Look,Depression
0,30.000000,10.000000
1,26.804183,9.407215
2,23.960180,8.756431
3,21.458817,8.051109
4,19.290917,7.294708


Calculate the look rate (dLook)

In [59]:
look_rate = np.diff(data.Look)

In [60]:
look_rate

array([-3.19581734, -2.84400257, -2.50136341, -2.16789985, -1.84361189,
       -1.52849953, -1.22256277, -0.92580161, -0.63821606, -0.35980611,
       -0.09057175,  0.169487  ,  0.42037015,  0.66207769,  0.89460964,
        1.11796599,  1.33214673,  1.53715187,  1.73298141,  1.91963535,
        2.09711369,  2.26541643,  2.42454356,  2.5744951 ,  2.71527103,
        2.84687136,  2.96929609,  3.08254522,  3.18661875,  3.28151668,
        3.367239  ,  3.44378572,  3.51115684,  3.56935237,  3.61837228,
        3.6582166 ,  3.68888532,  3.71037843,  3.72269595,  3.72583786,
        3.71980417,  3.70520179,  3.6852626 ,  3.66105811,  3.632589  ,
        3.59985528,  3.56285695,  3.52159402,  3.47606647,  3.42627431,
        3.37221755,  3.31389617,  3.25131019,  3.18445959,  3.11334439,
        3.03796458,  2.95832015,  2.87441112,  2.78623748,  2.69379923,
        2.59709637,  2.4961289 ,  2.39089682,  2.28140013,  2.16763883,
        2.04961292,  1.9273224 ,  1.80076727,  1.66994753,  1.53

Figure out where the data wraps back on itself (i.e. look_rate = 0)

In [66]:
look_rate_sign = look_rate > 0
look_rate_sign

array([False, False, False, False, False, False, False, False, False,
       False, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False])

Get the areas where the vertical asymptotes occur

In [80]:
asym_indices = np.where(
    look_rate_sign != np.roll(look_rate_sign, shift=1)
)[0].tolist()
print(asym_indices)

[11, 79]


# Display the transition points
These points are where the data should be split

In [86]:
my_frames = []
for idx in range(len(xnew) - 1):

    if idx in asym_indices:
        line_args = dict(color='red', width=4)
    else:
        line_args = dict(color='green', width=2)

    pt1 = xnew[idx], ynew[idx]
    pt2 = xnew[idx + 1], ynew[idx + 1]
    xs, ys = tangent_line(pt1, pt2, x_range=(0, 180))

    frame = go.Frame(data=[go.Scatter(x=xs, y=ys, mode='lines', line=line_args)])
    my_frames.append(frame)

In [89]:
fig = go.Figure(
    data=[
        go.Scatter(x=xnew, y=ynew, mode='lines', line=dict(width=2, color='blue')),
        go.Scatter(x=xnew, y=ynew, mode='lines', line=dict(width=2, color='blue'), showlegend=False)
    ],
    layout=go.Layout(
        xaxis=dict(range=[0, 200], autorange=False),
        yaxis=dict(range=[-45, 30], autorange=False),
        title="Fake Profile",
        updatemenus=[dict(
            type="buttons",
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[
                         None,
                         {
                             "frame": {"duration": 100, "redraw": False},
                             "mode": "immediate",
                             "transition": {"duration": 0}
                         }
                     ])
            ]
        )]
    ),
    frames=my_frames
)
fig.show()

In [81]:
# Need to add the start and end point of the data (aka 0 and data length) so that it can be properly sliced into subsections
asym_indices.insert(0, 0)
asym_indices.append(len(data))
print(asym_indices)

[0, 11, 79, 100]


# Split the original data frame into subsets

In [82]:
dataframes = {}
for subset_number, i in enumerate(range(len(asym_indices) - 1)):
    start_idx = asym_indices[i]
    end_idx = asym_indices[i+1]
    subset = data.iloc[start_idx:end_idx]
    dataframes[subset_number] = subset

In [94]:
for num, df in dataframes.items():
    print(f'Subset #{num+1}')
    print(df.head())

Subset #1
        Look  Depression
0  30.000000   10.000000
1  26.804183    9.407215
2  23.960180    8.756431
3  21.458817    8.051109
4  19.290917    7.294708
Subset #2
         Look  Depression
11  12.681847    0.860284
12  12.851334   -0.180207
13  13.271704   -1.240640
14  13.933782   -2.317557
15  14.828392   -3.407498
Subset #3
          Look  Depression
79  180.451391    2.630019
80  180.400857    3.610824
81  180.168329    4.569371
82  179.749540    5.503014
83  179.140227    6.409107


# Display the end result

In [96]:
from plotly.subplots import make_subplots

In [124]:
fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [{'colspan': 3}, None, None],
        [{}, {}, {}]
    ],
    shared_xaxes=True,
    shared_yaxes=True,
    subplot_titles=[
        'Original Profile',
        'Subset Profile 1', 'Subset Profile 2', 'Subset Profile 3'
    ]
)

original_profile = go.Scatter(x=xnew, y=ynew, mode='lines', line=dict(color='blue'), showlegend=False)
fig.add_trace(original_profile)

for num, df in dataframes.items():
    subset_profile = go.Scatter(x=df.Look, y=df.Depression, mode='lines', showlegend=False)
    fig.add_trace(subset_profile, row=2, col=num+1)

In [127]:
fig.update_layout(height=1000, xaxis=dict(title='Look'), yaxis=dict(title='Depression'))
fig.show()